In [2]:
from nemo.collections.asr.parts.submodules.ctc_beam_decoding import BeamCTCInfer
from inference_functions import load_bit_phoneme_model, evaluate_model
from dataset import getDatasetLoaders
import torch.nn.functional as F
import numpy as np
import torch
from edit_distance import SequenceMatcher

In [19]:
beam_search = BeamCTCInfer(blank_id=0, beam_size=16, ngram_lm_alpha=0.3, 
                           ngram_lm_model='/data/lm/lm_dec19_large_4gram.arpa')

with open("/data/lm/vocab_lower_100k.txt", "r", encoding="utf-8") as f:
    vocab = f.read().splitlines()
beam_search.set_vocabulary(vocab)
print("Vocabulary length: ", len(vocab))

[NeMo I 2025-08-31 00:22:26 ctc_beam_decoding:264] Beam search algorithm: default
Vocabulary length:  100002


In [20]:
device = 'cuda'
bit_phoneme_filepath = "/data/models/time_masked_transfomer_characters2_80ms_seed_0/"
model, args = load_bit_phoneme_model(bit_phoneme_filepath)
model = model.to(device)

data_file = '/data/neural_data/ptDecoder_ctc_both_char'
trainLoaders, testLoaders, loadedData = getDatasetLoaders(
        data_file, 8, None, 
        False
    )

outputs, cer, per_day_cer = evaluate_model(model, loadedData, args, partition='test', device='cuda')


CER DAY 0: 0.367578
CER DAY 1: 0.298714
CER DAY 2: 0.271638
CER DAY 3: 0.277852
CER DAY 4: 0.164248
CER DAY 5: 0.115663
CER DAY 6: 0.147847
CER DAY 7: 0.167688
CER DAY 8: 0.166441
CER DAY 9: 0.209104
CER DAY 10: 0.160420
CER DAY 11: 0.221338
CER DAY 12: 0.173742
CER DAY 13: 0.143961
CER DAY 14: 0.122901
CER DAY 15: 0.125095
CER DAY 16: 0.140399
CER DAY 17: 0.189916
CER DAY 18: 0.156134
CER DAY 19: 0.122260
CER DAY 20: 0.119109
CER DAY 21: 0.152846
CER DAY 22: 0.133858
CER DAY 23: 0.174182
Model performance (CER): 0.17518916065458384


In [21]:
num_classes = 33
logits = np.zeros((len(outputs['logits']), max(outputs['logitLengths']), num_classes))
for idx, l in enumerate(outputs['logits']):
    l_length = outputs['logitLengths'][idx]
    logits[idx, :l_length, :] = l
    
logits = torch.from_numpy(logits)
logit_lengths = torch.from_numpy(np.array(outputs['logitLengths']))

In [22]:
nbest = beam_search.flashlight_beam_search(logits, logit_lengths)